# Day 5 – Semantic Search

In [7]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util
import numpy as np
import re
import pandas as pd

In [8]:
sentences = [
    "Phishing emails trick users into revealing their passwords.",
    "A firewall blocks unauthorized network traffic.",
    "Ransomware encrypts files and demands payment to unlock them.",
    "Two-factor authentication adds an extra layer of login protection.",
    "Regular software updates fix security vulnerabilities.",
    "Hackers exploit weak passwords to break into accounts.",
    "Antivirus software detects and removes malicious programs.",
    "Encryption scrambles data so only authorized people can read it.",
    "A VPN hides your internet activity on public Wi-Fi.",
    "SQL injection attacks manipulate database queries through input fields.",
    "Employees should be trained to recognize social engineering attempts.",
    "Backups let organizations recover data after an attack.",
    "Penetration testers simulate attacks to find weaknesses before criminals do.",
    "A DDoS attack floods a website with traffic until it crashes.",
    "Password managers create and store strong, unique passwords.",
    "Intrusion detection systems monitor networks for suspicious activity.",
    "Zero-day vulnerabilities are flaws unknown to the software vendor.",
    "Biometric login uses fingerprints or face scans to verify identity.",
    "Malware can spread through infected USB drives and downloads.",
    "Security audits review whether a company follows safety policies.",
]
print(len(sentences), "sentences")

20 sentences


In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = model.encode(sentences)
print("Vector store shape:", corpus_embeddings.shape)   # (20, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store shape: (20, 384)


In [10]:
def semantic_search(query, top_k=3):
    q_emb = model.encode(query)
    scores = util.cos_sim(q_emb, corpus_embeddings)[0].numpy()
    top_idx = np.argsort(-scores)[:top_k]
    return [(round(float(scores[i]), 3), sentences[i]) for i in top_idx]

for score, text in semantic_search("How do I protect my account from being stolen?"):
    print(score, "|", text)

0.421 | Hackers exploit weak passwords to break into accounts.
0.399 | Password managers create and store strong, unique passwords.
0.347 | Phishing emails trick users into revealing their passwords.


In [11]:
stop = {"a","an","the","is","are","to","of","and","or","my","i","do","how","from",
        "in","it","that","by","be","being","with","for","on","so","can","their"}

def tokens(text):
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in stop}

def keyword_search(query, top_k=3):
    q = tokens(query)
    scored = [(len(q & tokens(s)), s) for s in sentences]
    scored = [x for x in scored if x[0] > 0]
    scored.sort(key=lambda x: -x[0])
    return scored[:top_k]

for score, text in keyword_search("How do I protect my account from being stolen?"):
    print(score, "|", text)

In [12]:
queries = [
    "How do I protect my account from being stolen?",
    "fake emails that steal login details",
    "someone locks my files and asks for money",
    "keeping my browsing private at a cafe",
]

for q in queries:
    print("=" * 70)
    print("QUERY:", q)
    print("\nSEMANTIC top 3:")
    for score, text in semantic_search(q):
        print(f"  {score}  {text}")
    print("\nKEYWORD top 3:")
    results = keyword_search(q)
    if not results:
        print("  No keyword matches found")
    for score, text in results:
        print(f"  {score} shared word(s)  {text}")

QUERY: How do I protect my account from being stolen?

SEMANTIC top 3:
  0.421  Hackers exploit weak passwords to break into accounts.
  0.399  Password managers create and store strong, unique passwords.
  0.347  Phishing emails trick users into revealing their passwords.

KEYWORD top 3:
  No keyword matches found
QUERY: fake emails that steal login details

SEMANTIC top 3:
  0.625  Phishing emails trick users into revealing their passwords.
  0.435  Hackers exploit weak passwords to break into accounts.
  0.39  Biometric login uses fingerprints or face scans to verify identity.

KEYWORD top 3:
  1 shared word(s)  Phishing emails trick users into revealing their passwords.
  1 shared word(s)  Two-factor authentication adds an extra layer of login protection.
  1 shared word(s)  Biometric login uses fingerprints or face scans to verify identity.
QUERY: someone locks my files and asks for money

SEMANTIC top 3:
  0.623  Ransomware encrypts files and demands payment to unlock them.
  0.3

## Reflection

Semantic search handled my queries better than keyword search. For "someone locks my files and asks for money", it returned the ransomware sentence as the top result even though the query never used the word "ransomware". Keyword search only matched on common words like "files", or returned nothing useful. For queries that shared exact words with a sentence, both methods worked about equally well. This surprised me because the model understood the meaning of the query, not just its words, which is why semantic search is used in RAG and AI-powered Q&A.